# 04 - ETL Bronze → Silver (PostgreSQL)

**Architecture Medallion – Couche Silver**

Ce notebook charge les données brutes (Bronze) et les insère dans la base PostgreSQL `mspr813`, schéma `silver`.

| Table | Source Bronze | Lignes attendues |
|-------|--------------|------------------|
| `silver.referentiel_communes` | referentiels_cog/referentiel_communes_2024.csv | ~144 |
| `silver.elections` | elections_agregees_1999_2024.csv | ~1.5M |
| `silver.naissances` | naissances_2008_2024/DS_ETAT_CIVIL_NAIS_COMMUNES_data.csv | ~2000 |
| `silver.deces` | deces_2008_2024/DS_ETAT_CIVIL_DECES_COMMUNES_data.csv | ~2000 |
| `silver.revenus` | revenus_commune.csv | ~1500 |
| `silver.csp` | csp_actifs_2554/pop-act2554-csp-cd-6822.xlsx | ~144 |
| `silver.diplomes` | diplomes_formation_2022/base-cc-diplomes-formation-2022.xlsx | ~144 |

## 0. Configuration & imports

In [6]:
import os
import re
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# --- Configuration inline (pas d'import externe, compatible VS Code Jupyter) ---
BRONZE_DIR = "/app/data/bronze"

# Connexion PostgreSQL (variables d'environnement injectées par docker-compose)
PG_HOST     = os.environ.get("POSTGRES_HOST",     "postgres")
PG_PORT     = os.environ.get("POSTGRES_PORT",     "5432")
PG_DB       = os.environ.get("POSTGRES_DB",       "mspr813")
PG_USER     = os.environ.get("POSTGRES_USER",     "mspr_user")
PG_PASSWORD = os.environ.get("POSTGRES_PASSWORD", "mspr_password")

DB_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# Départements Petite Couronne
DEPS_PC = ['75', '92', '93', '94']

engine = create_engine(DB_URL, pool_pre_ping=True)
print(f"Connexion : {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# Test rapide
with engine.connect() as conn:
    r = conn.execute(text("SELECT current_database(), current_schema()"))
    print("PostgreSQL OK :", r.fetchone())

Connexion : mspr_user@postgres:5432/mspr813
PostgreSQL OK : ('mspr813', 'public')


## 1. Référentiel communes (COG 2024)
Source : `referentiels_cog/referentiel_communes_2024.csv`

In [7]:
cog_path = os.path.join(BRONZE_DIR, "referentiels_cog", "referentiel_communes_2024.csv")
df_cog_raw = pd.read_csv(cog_path, sep=',', dtype=str, low_memory=False)
print("Colonnes:", df_cog_raw.columns.tolist())
print("Shape:", df_cog_raw.shape)
df_cog_raw.head(3)

Colonnes: ['TYPECOM', 'COM', 'REG', 'DEP', 'CTCD', 'ARR', 'TNCC', 'NCC', 'NCCENR', 'LIBELLE', 'CAN', 'COMPARENT']
Shape: (37544, 12)


,TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT
0,COM,01001,84,01,01D,012,5,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,L'Abergement-Clémenciat,0108,NaN
1,COM,01002,84,01,01D,011,5,ABERGEMENT DE VAREY,Abergement-de-Varey,L'Abergement-de-Varey,0101,NaN
2,COM,01004,84,01,01D,011,1,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,Ambérieu-en-Bugey,0101,NaN


In [8]:
# Identifier la colonne code INSEE et libellé
# Colonnes attendues : COM (ou CODGEO), LIBELLE (ou NCC), DEP, REG
col_map = {c.upper(): c for c in df_cog_raw.columns}
print("Colonnes normalisées:", list(col_map.keys()))

Colonnes normalisées: ['TYPECOM', 'COM', 'REG', 'DEP', 'CTCD', 'ARR', 'TNCC', 'NCC', 'NCCENR', 'LIBELLE', 'CAN', 'COMPARENT']


In [9]:
# Nettoyage et filtrage Petite Couronne
df_cog = df_cog_raw.copy()

# Renommer les colonnes clés (adaptation selon le fichier réel)
rename = {}
for src, tgt in [('COM','code_insee'), ('CODGEO','code_insee'),
                  ('LIBELLE','libelle'), ('NCC','libelle'),
                  ('DEP','code_dep'), ('REG','code_reg'),
                  ('STATUT','statut'), ('ARR','arr')]:
    if src in col_map:
        rename[col_map[src]] = tgt

df_cog = df_cog.rename(columns=rename)

# Supprimer les colonnes dupliquées (ex: LIBELLE et NCC renommées toutes deux en 'libelle')
df_cog = df_cog.loc[:, ~df_cog.columns.duplicated()]

print("Colonnes après renommage:", df_cog.columns.tolist())

# Garder seulement les colonnes Silver
keep = ['code_insee', 'libelle', 'code_dep', 'code_reg', 'statut', 'arr']
existing = [c for c in keep if c in df_cog.columns]
df_cog = df_cog[existing].copy()

# Pad code_insee à 5 chars
df_cog['code_insee'] = df_cog['code_insee'].astype(str).str.zfill(5)

# Filtrer Petite Couronne (dep 75, 92, 93, 94)
# code_dep peut être '075' ou '75' selon le fichier
df_cog['code_dep_norm'] = df_cog['code_dep'].astype(str).str.lstrip('0').str.zfill(2)
df_cog = df_cog[df_cog['code_dep_norm'].isin(DEPS_PC)].copy()
df_cog = df_cog.drop(columns=['code_dep_norm'])

print(f"Communes Petite Couronne : {len(df_cog)}")
df_cog.head(5)


Colonnes après renommage: ['TYPECOM', 'code_insee', 'code_reg', 'code_dep', 'CTCD', 'arr', 'TNCC', 'libelle', 'NCCENR', 'CAN', 'COMPARENT']
Communes Petite Couronne : 144


,code_insee,libelle,code_dep,code_reg,arr
31482,75056,PARIS,75,11,751
31483,75101,PARIS 1ER ARRONDISSEMENT,75,11,751
31484,75102,PARIS 2E ARRONDISSEMENT,75,11,751
31485,75103,PARIS 3E ARRONDISSEMENT,75,11,751
31486,75104,PARIS 4E ARRONDISSEMENT,75,11,751


In [10]:
# Insertion PostgreSQL
with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.referentiel_communes CASCADE"))

df_cog.to_sql(
    'referentiel_communes',
    engine,
    schema='silver',
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.referentiel_communes")).scalar()
print(f"silver.referentiel_communes : {n} lignes insérées")

silver.referentiel_communes : 144 lignes insérées


## 2. Elections 1999–2024
Source : `elections_agregees_1999_2024.csv` (~2.35 GB, ~1.5M lignes)

Attention : lecture par chunks pour éviter OOM.

In [11]:
elec_path = os.path.join(BRONZE_DIR, "elections_agregees_1999_2024.csv")

# Lire 5 premières lignes pour identifier les colonnes
df_elec_sample = pd.read_csv(elec_path, sep=';', nrows=5, dtype=str)
print("Colonnes:", df_elec_sample.columns.tolist())
df_elec_sample

Colonnes: ['id_election', 'id_brut_miom', 'code_departement', 'code_commune', 'code_bv', 'no_panneau', 'voix', 'ratio_voix_inscrits', 'ratio_voix_exprimes', 'nuance', 'sexe', 'nom', 'prenom', 'liste', 'libelle_abrege_liste', 'libelle_etendu_liste', 'nom_tete_liste', 'binome']


,id_election,id_brut_miom,code_departement,code_commune,code_bv,no_panneau,voix,ratio_voix_inscrits,ratio_voix_exprimes,nuance,sexe,nom,prenom,liste,libelle_abrege_liste,libelle_etendu_liste,nom_tete_liste,binome
0,2022_pres_t1,01001_0001,01,01001,0001,1,3,0.47,0.58,NaN,F,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN
1,2022_pres_t1,01002_0001,01,01002,0001,1,2,0.94,1.17,NaN,F,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN
2,2022_pres_t1,01004_0001,01,01004,0001,1,4,0.35,0.48,NaN,F,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN
3,2022_pres_t1,01004_0002,01,01004,0002,1,6,0.53,0.71,NaN,F,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN
4,2022_pres_t1,01004_0003,01,01004,0003,1,8,0.66,0.84,NaN,F,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN


In [12]:
# Obtenir la liste des codes communes PC depuis le référentiel
with engine.connect() as conn:
    codes_pc = set(
        r[0] for r in conn.execute(text("SELECT code_insee FROM silver.referentiel_communes"))
    )
print(f"Codes communes PC chargés : {len(codes_pc)}")

# Vider la table elections
with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.elections"))
print("Table silver.elections vidée")

Codes communes PC chargés : 144
Table silver.elections vidée


In [13]:
# Mapping colonnes Bronze -> Silver (à adapter selon les colonnes réelles)
# On va détecter dynamiquement
col_elec = df_elec_sample.columns.tolist()
print("Colonnes élections:", col_elec)

# Regex pour extraire l'année depuis id_election (ex: '2022_pres_t1' -> 2022)
def extract_year(id_election):
    m = re.search(r'(\d{4})', str(id_election))
    return int(m.group(1)) if m else None

def extract_type(id_election):
    """Extrait le type ex: 'pres', 'leg', 'mun' depuis '2022_pres_t1'"""
    parts = str(id_election).split('_')
    return parts[1] if len(parts) >= 2 else None

def extract_tour(id_election):
    m = re.search(r't(\d)', str(id_election))
    return int(m.group(1)) if m else None

Colonnes élections: ['id_election', 'id_brut_miom', 'code_departement', 'code_commune', 'code_bv', 'no_panneau', 'voix', 'ratio_voix_inscrits', 'ratio_voix_exprimes', 'nuance', 'sexe', 'nom', 'prenom', 'liste', 'libelle_abrege_liste', 'libelle_etendu_liste', 'nom_tete_liste', 'binome']


In [14]:
# Lecture par chunks + filtrage PC + insertion PostgreSQL
CHUNK_SIZE = 50_000
total_rows = 0
total_chunks = 0

# Correspondance colonnes Bronze -> noms Silver
# Ces noms doivent correspondre aux colonnes réelles du fichier
ELEC_COL_REMAP = {
    'code_commune':    'code_commune',
    'id_election':     'id_election',
    'nuance_liste':    'nuance_liste',
    'nom_liste':       'nom_liste',
    'nom_candidat':    'nom_candidat',
    'prenom_candidat': 'prenom_candidat',
    'sexe':            'sexe',
    'nb_voix':         'nb_voix',
    'pct_voix_ins':    'pct_voix_ins',
    'pct_voix_exp':    'pct_voix_exp',
    'inscrits':        'inscrits',
    'votants':         'votants',
    'exprimes':        'exprimes',
}

reader = pd.read_csv(
    elec_path,
    sep=';',
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
)

for chunk in reader:
    # Normaliser code_commune
    chunk['code_commune'] = chunk['code_commune'].astype(str).str.zfill(5)
    
    # Filtrer Petite Couronne
    chunk = chunk[chunk['code_commune'].isin(codes_pc)].copy()
    
    if chunk.empty:
        total_chunks += 1
        continue
    
    # Ajouter colonnes dérivées
    chunk['annee']         = chunk['id_election'].apply(extract_year).astype('Int16')
    chunk['type_election'] = chunk['id_election'].apply(extract_type)
    chunk['tour']          = chunk['id_election'].apply(extract_tour).astype('Int8')
    
    # Sélectionner et renommer les colonnes Silver disponibles
    silver_cols = ['code_commune', 'id_election', 'annee', 'type_election', 'tour']
    for bronze_col, silver_col in ELEC_COL_REMAP.items():
        if bronze_col in chunk.columns and bronze_col not in silver_cols:
            silver_cols.append(bronze_col)
    
    chunk = chunk[[c for c in silver_cols if c in chunk.columns]].copy()
    
    # Convertir types numériques
    for col in ['nb_voix', 'inscrits', 'votants', 'exprimes']:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('Int64')
    for col in ['pct_voix_ins', 'pct_voix_exp']:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col].str.replace(',', '.', regex=False), errors='coerce')
    
    # Insérer
    chunk.to_sql(
        'elections',
        engine,
        schema='silver',
        if_exists='append',
        index=False,
        method='multi',
        chunksize=2000
    )
    
    total_rows += len(chunk)
    total_chunks += 1
    
    if total_chunks % 10 == 0:
        print(f"  Chunk {total_chunks} | PC insérées cumulées : {total_rows:,}")

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.elections")).scalar()
print(f"\nsilver.elections : {n:,} lignes totales")

  Chunk 10 | PC insérées cumulées : 24,878
  Chunk 20 | PC insérées cumulées : 49,724
  Chunk 30 | PC insérées cumulées : 74,490
  Chunk 40 | PC insérées cumulées : 100,153
  Chunk 50 | PC insérées cumulées : 127,560
  Chunk 60 | PC insérées cumulées : 152,326
  Chunk 70 | PC insérées cumulées : 177,152
  Chunk 80 | PC insérées cumulées : 209,138
  Chunk 90 | PC insérées cumulées : 233,717
  Chunk 100 | PC insérées cumulées : 270,722
  Chunk 110 | PC insérées cumulées : 295,362
  Chunk 130 | PC insérées cumulées : 348,590
  Chunk 140 | PC insérées cumulées : 387,070
  Chunk 150 | PC insérées cumulées : 422,586
  Chunk 160 | PC insérées cumulées : 447,870
  Chunk 170 | PC insérées cumulées : 473,154
  Chunk 180 | PC insérées cumulées : 498,438
  Chunk 190 | PC insérées cumulées : 523,722
  Chunk 200 | PC insérées cumulées : 549,006
  Chunk 210 | PC insérées cumulées : 576,765
  Chunk 260 | PC insérées cumulées : 705,745
  Chunk 350 | PC insérées cumulées : 978,104
  Chunk 360 | PC insér

## 3. Naissances 2008–2024

In [15]:
nais_path = os.path.join(BRONZE_DIR, "naissances_2008_2024", "DS_ETAT_CIVIL_NAIS_COMMUNES_data.csv")
df_nais_raw = pd.read_csv(nais_path, sep=';', dtype=str, low_memory=False)
print("Colonnes:", df_nais_raw.columns.tolist())
print("Shape:", df_nais_raw.shape)
df_nais_raw.head(3)

Colonnes: ['EC_MEASURE', 'FREQ', 'GEO', 'GEO_OBJECT', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE']
Shape: (710821, 7)


,EC_MEASURE,FREQ,GEO,GEO_OBJECT,OBS_STATUS,TIME_PERIOD,OBS_VALUE
0,LVB,A,68170,COM,A,2018,5
1,LVB,A,68185,COM,A,2015,14
2,LVB,A,68181,COM,A,2018,1


In [16]:
# Colonnes attendues : GEO, TIME_PERIOD, OBS_VALUE, EC_MEASURE
# Filtrer mesure = nombre de naissances (EC_MEASURE)
print("EC_MEASURE valeurs uniques:", df_nais_raw['EC_MEASURE'].unique() if 'EC_MEASURE' in df_nais_raw.columns else 'colonne absente')

EC_MEASURE valeurs uniques: ['LVB']


In [17]:
df_nais = df_nais_raw.copy()

# Renommer les colonnes
df_nais = df_nais.rename(columns={
    'GEO':         'code_commune',
    'TIME_PERIOD': 'annee',
    'OBS_VALUE':   'nb_naissances'
})

# Filtrer uniquement la mesure 'nombre' si EC_MEASURE existe
if 'EC_MEASURE' in df_nais.columns:
    # Garder la mesure principale (souvent la seule ou la plus complète)
    measures = df_nais['EC_MEASURE'].unique()
    print("Mesures disponibles:", measures)
    # Prendre la première valeur si une seule, sinon filtrer sur 'NOMBRE' ou similaire
    if len(measures) == 1:
        df_nais = df_nais[['code_commune', 'annee', 'nb_naissances']]
    else:
        # Filtrer sur la mesure qui ressemble à un total
        for m in ['NOMBRE', 'T', 'TOTAL', 'NAIS']:
            if m in measures:
                df_nais = df_nais[df_nais['EC_MEASURE'] == m]
                break
        df_nais = df_nais[['code_commune', 'annee', 'nb_naissances']]

# Pad code_commune
df_nais['code_commune'] = df_nais['code_commune'].astype(str).str.zfill(5)

# Filtrer PC
df_nais = df_nais[df_nais['code_commune'].isin(codes_pc)].copy()

# Types
df_nais['annee'] = pd.to_numeric(df_nais['annee'], errors='coerce').astype('Int16')
df_nais['nb_naissances'] = pd.to_numeric(df_nais['nb_naissances'], errors='coerce').astype('Int32')

# Dédupliquer
df_nais = df_nais.drop_duplicates(subset=['code_commune', 'annee'])

print(f"Naissances PC : {len(df_nais)} lignes")
df_nais.head(5)

Mesures disponibles: ['LVB']
Naissances PC : 2431 lignes


,code_commune,annee,nb_naissances
17583,75056,2024,21484
18412,75056,2010,31447
18628,75056,2013,28945
19189,75056,2012,29291
19745,75056,2020,26157


In [18]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.naissances"))

df_nais.to_sql(
    'naissances',
    engine,
    schema='silver',
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.naissances")).scalar()
print(f"silver.naissances : {n} lignes")

silver.naissances : 2431 lignes


## 4. Décès 2008–2024

In [19]:
deces_path = os.path.join(BRONZE_DIR, "deces_2008_2024", "DS_ETAT_CIVIL_DECES_COMMUNES_data.csv")
df_deces_raw = pd.read_csv(deces_path, sep=';', dtype=str, low_memory=False)
print("Colonnes:", df_deces_raw.columns.tolist())
print("Shape:", df_deces_raw.shape)

df_deces = df_deces_raw.copy()
df_deces = df_deces.rename(columns={
    'GEO':         'code_commune',
    'TIME_PERIOD': 'annee',
    'OBS_VALUE':   'nb_deces'
})

if 'EC_MEASURE' in df_deces.columns:
    measures = df_deces['EC_MEASURE'].unique()
    if len(measures) > 1:
        for m in ['NOMBRE', 'T', 'TOTAL', 'DECES']:
            if m in measures:
                df_deces = df_deces[df_deces['EC_MEASURE'] == m]
                break
    df_deces = df_deces[['code_commune', 'annee', 'nb_deces']]

df_deces['code_commune'] = df_deces['code_commune'].astype(str).str.zfill(5)
df_deces = df_deces[df_deces['code_commune'].isin(codes_pc)].copy()
df_deces['annee']    = pd.to_numeric(df_deces['annee'], errors='coerce').astype('Int16')
df_deces['nb_deces'] = pd.to_numeric(df_deces['nb_deces'], errors='coerce').astype('Int32')
df_deces = df_deces.drop_duplicates(subset=['code_commune', 'annee'])

print(f"Décès PC : {len(df_deces)} lignes")

with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.deces"))

df_deces.to_sql('deces', engine, schema='silver', if_exists='append', index=False, method='multi', chunksize=1000)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.deces")).scalar()
print(f"silver.deces : {n} lignes")

Colonnes: ['EC_MEASURE', 'FREQ', 'GEO', 'GEO_OBJECT', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE']
Shape: (710821, 7)
Décès PC : 2431 lignes
silver.deces : 2431 lignes


## 5. Revenus par commune

In [20]:
rev_path = os.path.join(BRONZE_DIR, "revenus_commune.csv")
df_rev_raw = pd.read_csv(rev_path, sep=';', dtype=str, low_memory=False)
print("Colonnes (30 premières):", df_rev_raw.columns.tolist()[:30])
print("Shape:", df_rev_raw.shape)
df_rev_raw.head(3)

Colonnes (30 premières): ['Nom géographique GMS', 'Code géographique', 'Libellé géographique', '[DISP] Nbre de ménages fiscaux', '[DISP] Nbre de personnes dans les ménages fiscaux', "[DISP] Nbre d'unités de consommation dans les ménages fiscaux", '[DISP] 1ᵉʳ quartile (€)', '[DISP] Médiane (€)', '[DISP] 3ᵉ quartile (€)', '[DISP] Écart interquartile (€)', '[DISP] 1ᵉʳ décile (€)', '[DISP] 2ᵉ décile (€)', '[DISP]3ᵉ décile (€)', '[DISP] 4ᵉ décile (€)', '[DISP] 6ᵉ décile (€)', '[DISP] 7ᵉ décile (€)', '[DISP] 8ᵉ décile (€)', '[DISP] 9ᵉ décile (€)', '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile', '[DISP] S80/20', '[DISP] Iice de Gini', '[DISP] Part des revenus d’activité (%)', '[DISP] dont part des salaires et traitements(%)', '[DISP] dont part des iemnités de chômage (%)', '[DISP] dont part des revenus des activités non salariées (%)', '[DISP] Part des pensions, retraites et rentes (%)', '[DISP] Part des revenus du patrimoine et autres revenus (%)', "[DISP] Part de l'ensemble des prestatio

,Nom géographique GMS,Code géographique,Libellé géographique,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Nbre d'unités de consommation dans les ménages fiscaux,[DISP] 1ᵉʳ quartile (€),[DISP] Médiane (€),[DISP] 3ᵉ quartile (€),[DISP] Écart interquartile (€),...,[DEC] 9ᵉ décile (€),[DEC] Rapport interdécile D9/D1,[DEC] S80/S20,[DEC] Iice de Gini,[DEC] Part des revenus d’activité (%),[DEC] dont part des salaires et traitements (%),[DEC] dont part des iemnités de chômage (%),[DEC] dont part des revenus des activités non salariées (%),"[DEC] Part des pensions, retraites et rentes (%)",[DEC] Part des autres revenus (%)
0,L'Abergement-Clémenciat,01001,L'Abergement-Clémenciat,346,895,590.8,NaN,25820,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L'Abergement-de-Varey,01002,L'Abergement-de-Varey,115,266,181,NaN,24480,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ambérieu-en-Bugey,01004,Ambérieu-en-Bugey,6855,15092,10398.2,15800,21660,28430,12630,...,39380,"5,6",8.1,0.368,67.9,61.5,3.2,3.2,24.4,7.7


In [21]:
# Identifier les colonnes clés
# code commune : 'Code géographique' ou 'CODGEO'
# annee : à détecter (peut être dans le nom de colonne)
cols = df_rev_raw.columns.tolist()

# Chercher colonne commune
code_col = None
for c in cols:
    if 'géographique' in c.lower() or 'codgeo' in c.lower() or c.upper() == 'CODE':
        code_col = c
        break
print(f"Colonne code commune : {code_col}")

# Chercher colonnes revenus - format [DISP] et [DEC]
disp_cols = [c for c in cols if '[DISP]' in c or 'DISP' in c.upper()]
dec_cols  = [c for c in cols if '[DEC]' in c  or 'DEC' in c.upper()]
print(f"Colonnes DISP ({len(disp_cols)}): {disp_cols[:5]}")
print(f"Colonnes DEC  ({len(dec_cols)}):  {dec_cols[:5]}")

Colonne code commune : Nom géographique GMS
Colonnes DISP (29): ['[DISP] Nbre de ménages fiscaux', '[DISP] Nbre de personnes dans les ménages fiscaux', "[DISP] Nbre d'unités de consommation dans les ménages fiscaux", '[DISP] 1ᵉʳ quartile (€)', '[DISP] Médiane (€)']
Colonnes DEC  (26):  ['[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile', '[DEC] Nbre de ménages fiscaux', '[DEC] Nbre de personnes dans les ménages fiscaux', "[DEC] Nbre d'unités de consommation dans les ménages fiscaux", '[DEC] Part des ménages fiscaux imposés (%)']


In [22]:
# Construire le DataFrame revenus
# Les colonnes peuvent inclure l'année dans leur nom (ex: '[DISP] D5 2019')
# On va pivoter pour avoir une ligne par commune+année

df_rev = df_rev_raw.copy()

if code_col:
    df_rev = df_rev.rename(columns={code_col: 'code_commune'})
else:
    # Essayer la première colonne
    df_rev = df_rev.rename(columns={cols[0]: 'code_commune'})

df_rev['code_commune'] = df_rev['code_commune'].astype(str).str.zfill(5)
df_rev = df_rev[df_rev['code_commune'].isin(codes_pc)].copy()

print(f"Communes PC trouvées : {df_rev['code_commune'].nunique()}")

# Détecter si les colonnes contiennent des années
year_pattern = re.compile(r'(\d{4})')
years_in_cols = set()
for c in df_rev.columns:
    m = year_pattern.search(c)
    if m:
        years_in_cols.add(int(m.group(1)))
years_in_cols = sorted(years_in_cols)
print(f"Années détectées dans les colonnes : {years_in_cols}")

Communes PC trouvées : 0
Années détectées dans les colonnes : []


In [23]:
# Construire le DataFrame Silver revenus
# Si plusieurs années dans les colonnes, on fait un pivot long
# Si une seule année ou pas d'années, on met une année par défaut

silver_rev_rows = []

if years_in_cols:
    for year in years_in_cols:
        yr = str(year)
        # Colonnes correspondant à cette année
        def find_col(keywords):
            for kw in keywords:
                for c in df_rev.columns:
                    if yr in c and kw.upper() in c.upper():
                        return c
            return None

        c_mediane_disp = find_col(['D5', 'MED', 'MEDI'])
        c_q1           = find_col(['D1'])
        c_q9           = find_col(['D9'])
        c_gini         = find_col(['GINI', 'GINI'])
        c_pauvrete     = find_col(['TPAUV', 'PAUVRE', 'PAUV'])
        c_mediane_dec  = find_col(['DEC'])

        tmp = df_rev[['code_commune']].copy()
        tmp['annee'] = year

        for silver_col, bronze_col in [
            ('mediane_revenu_disp', c_mediane_disp),
            ('q1_revenu_disp',      c_q1),
            ('q9_revenu_disp',      c_q9),
            ('gini',               c_gini),
            ('taux_pauvrete',      c_pauvrete),
            ('mediane_revenu_dec', c_mediane_dec),
        ]:
            if bronze_col and bronze_col in df_rev.columns:
                tmp[silver_col] = pd.to_numeric(
                    df_rev[bronze_col].str.replace(',', '.', regex=False),
                    errors='coerce'
                ).values
            else:
                tmp[silver_col] = np.nan

        silver_rev_rows.append(tmp)

    df_rev_silver = pd.concat(silver_rev_rows, ignore_index=True)
else:
    # Pas d'année dans les colonnes : créer avec une seule année (la plus récente disponible)
    df_rev_silver = df_rev[['code_commune']].copy()
    df_rev_silver['annee'] = 2022
    for col in ['mediane_revenu_disp', 'q1_revenu_disp', 'q9_revenu_disp',
                'gini', 'taux_pauvrete', 'mediane_revenu_dec']:
        df_rev_silver[col] = np.nan

# Dédupliquer
df_rev_silver = df_rev_silver.drop_duplicates(subset=['code_commune', 'annee'])
df_rev_silver['annee'] = df_rev_silver['annee'].astype('Int16')

print(f"Revenus Silver : {len(df_rev_silver)} lignes, {df_rev_silver['code_commune'].nunique()} communes")
df_rev_silver.head(3)

Revenus Silver : 0 lignes, 0 communes


,code_commune,annee,mediane_revenu_disp,q1_revenu_disp,q9_revenu_disp,gini,taux_pauvrete,mediane_revenu_dec


In [24]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.revenus"))

df_rev_silver.to_sql('revenus', engine, schema='silver', if_exists='append', index=False, method='multi', chunksize=1000)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.revenus")).scalar()
print(f"silver.revenus : {n} lignes")

silver.revenus : 0 lignes


## 6. CSP actifs 25-54 ans (RP 2022)
Source : `csp_actifs_2554/pop-act2554-csp-cd-6822.xlsx`
- Sheet : `COM_2022`
- Header : row 14 (humain) / row 15 (codes techniques)
- Données : row 16+
- Code INSEE : `DR` (dept 2 chars) + `CR` (commune 3 chars)

In [25]:
csp_path = os.path.join(BRONZE_DIR, "csp_actifs_2554", "pop-act2554-csp-cd-6822.xlsx")

# skiprows=15 pour avoir la row 15 (codes techniques) comme header
# (row 0-14 skippées, row 15 = header)
df_csp_raw = pd.read_excel(
    csp_path,
    sheet_name='COM_2022',
    skiprows=15,
    header=0,
    dtype=str,
    engine='openpyxl'
)
print("Colonnes:", df_csp_raw.columns.tolist())
print("Shape:", df_csp_raw.shape)
df_csp_raw.head(3)

Colonnes: ['RR', 'DR', 'CR', 'STABLE', 'DR24', 'LIBELLE', 'csx_rec1taxtypac_rec1rpop2022', 'csx_rec1taxtypac_rec2rpop2022', 'csx_rec2taxtypac_rec1rpop2022', 'csx_rec2taxtypac_rec2rpop2022', 'csx_rec3taxtypac_rec1rpop2022', 'csx_rec3taxtypac_rec2rpop2022', 'csx_rec4taxtypac_rec1rpop2022', 'csx_rec4taxtypac_rec2rpop2022', 'csx_rec5taxtypac_rec1rpop2022', 'csx_rec5taxtypac_rec2rpop2022', 'csx_rec6taxtypac_rec1rpop2022', 'csx_rec6taxtypac_rec2rpop2022']
Shape: (38214, 18)


,RR,DR,CR,STABLE,DR24,LIBELLE,csx_rec1taxtypac_rec1rpop2022,csx_rec1taxtypac_rec2rpop2022,csx_rec2taxtypac_rec1rpop2022,csx_rec2taxtypac_rec2rpop2022,csx_rec3taxtypac_rec1rpop2022,csx_rec3taxtypac_rec2rpop2022,csx_rec4taxtypac_rec1rpop2022,csx_rec4taxtypac_rec2rpop2022,csx_rec5taxtypac_rec1rpop2022,csx_rec5taxtypac_rec2rpop2022,csx_rec6taxtypac_rec1rpop2022,csx_rec6taxtypac_rec2rpop2022
0,84,01,001,1,01,L'Abergement-Clémenciat,0,0,14.7257142857143,0,53.9942857142857,0,88.3542857142858,0,88.3542857142858,0,63.8114285714286,4.90857142857143
1,84,01,002,1,01,L'Abergement-de-Varey,24.8181818181818,0,4.96363636363636,0,9.92727272727272,0,19.8545454545454,0,14.8909090909091,4.96363636363636,14.8909090909091,0
2,84,01,003,0,01,Amareins,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_csp = df_csp_raw.copy()

# Construire code INSEE = DR (padded 2) + CR (padded 3)
df_csp['code_commune'] = (
    df_csp['DR'].astype(str).str.zfill(2) +
    df_csp['CR'].astype(str).str.zfill(3)
)

# Filtrer PC
df_csp = df_csp[df_csp['code_commune'].isin(codes_pc)].copy()
df_csp['annee'] = 2022

# Mappage colonnes techniques -> Silver
# csx_recXtaxtypac_rec1 = actifs emploi, rec2 = chômeurs
col_mapping = {
    'csx_rec1taxtypac_rec1rpop2022': 'agriculteurs_emploi',
    'csx_rec1taxtypac_rec2rpop2022': 'agriculteurs_chomeurs',
    'csx_rec2taxtypac_rec1rpop2022': 'artisans_emploi',
    'csx_rec2taxtypac_rec2rpop2022': 'artisans_chomeurs',
    'csx_rec3taxtypac_rec1rpop2022': 'cadres_emploi',
    'csx_rec3taxtypac_rec2rpop2022': 'cadres_chomeurs',
    'csx_rec4taxtypac_rec1rpop2022': 'prof_interm_emploi',
    'csx_rec4taxtypac_rec2rpop2022': 'prof_interm_chomeurs',
    'csx_rec5taxtypac_rec1rpop2022': 'employes_emploi',
    'csx_rec5taxtypac_rec2rpop2022': 'employes_chomeurs',
    'csx_rec6taxtypac_rec1rpop2022': 'ouvriers_emploi',
    'csx_rec6taxtypac_rec2rpop2022': 'ouvriers_chomeurs',
}

df_csp_silver = df_csp[['code_commune', 'annee']].copy()
for bronze_col, silver_col in col_mapping.items():
    if bronze_col in df_csp.columns:
        df_csp_silver[silver_col] = pd.to_numeric(df_csp[bronze_col], errors='coerce')
    else:
        print(f"WARN: colonne {bronze_col} absente")
        df_csp_silver[silver_col] = np.nan

df_csp_silver['annee'] = df_csp_silver['annee'].astype('Int16')
df_csp_silver = df_csp_silver.drop_duplicates(subset=['code_commune', 'annee'])

print(f"CSP Silver : {len(df_csp_silver)} communes")
df_csp_silver.head(3)

CSP Silver : 143 communes


,code_commune,annee,agriculteurs_emploi,agriculteurs_chomeurs,artisans_emploi,artisans_chomeurs,cadres_emploi,cadres_chomeurs,prof_interm_emploi,prof_interm_chomeurs,employes_emploi,employes_chomeurs,ouvriers_emploi,ouvriers_chomeurs
32121,75101,2022,0.0,0.0,571.869890,13.059980,3650.766466,197.038083,1200.553600,139.168686,719.464827,94.407113,197.075952,32.508638
32122,75102,2022,0.0,0.0,554.444000,16.873390,5658.461498,361.159783,1679.219825,211.289279,872.218035,191.786885,332.081452,59.991698
32123,75103,2022,0.0,0.0,990.608042,30.272866,8032.241981,477.288580,2608.086434,290.456070,1500.052049,236.964562,403.895404,134.435775


In [27]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.csp"))

df_csp_silver.to_sql('csp', engine, schema='silver', if_exists='append', index=False, method='multi', chunksize=1000)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.csp")).scalar()
print(f"silver.csp : {n} lignes")

silver.csp : 143 lignes


## 7. Diplômes et formation (RP 2022)
Source : `diplomes_formation_2022/base-cc-diplomes-formation-2022.xlsx`
- Sheet : `COM_2022`
- `skiprows=5`, header = row 5 (codes CODGEO, P22_...)

In [28]:
dipl_path = os.path.join(BRONZE_DIR, "diplomes_formation_2022", "base-cc-diplomes-formation-2022.xlsx")

df_dipl_raw = pd.read_excel(
    dipl_path,
    sheet_name='COM_2022',
    skiprows=5,
    header=0,
    dtype=str,
    engine='openpyxl'
)
print("Colonnes (15 premières):", df_dipl_raw.columns.tolist()[:15])
print("Shape:", df_dipl_raw.shape)
df_dipl_raw.head(3)

Colonnes (15 premières): ['CODGEO', 'REG', 'DEP', 'LIBGEO', 'P22_POP0205', 'P22_POP0610', 'P22_POP1114', 'P22_POP1517', 'P22_POP1824', 'P22_POP2529', 'P22_POP30P', 'P22_SCOL0205', 'P22_SCOL0610', 'P22_SCOL1114', 'P22_SCOL1517']
Shape: (34858, 70)


,CODGEO,REG,DEP,LIBGEO,P22_POP0205,P22_POP0610,P22_POP1114,P22_POP1517,P22_POP1824,P22_POP2529,...,P22_HNSCOL15P_SUP34,P22_HNSCOL15P_SUP5,P22_FNSCOL15P,P22_FNSCOL15P_DIPLMIN,P22_FNSCOL15P_BEPC,P22_FNSCOL15P_CAPBEP,P22_FNSCOL15P_BAC,P22_FNSCOL15P_SUP2,P22_FNSCOL15P_SUP34,P22_FNSCOL15P_SUP5
0,01001,84,01,L'Abergement-Clémenciat,34,56,45,36,30,37,...,23,20,331,57,17,82,67,47,40,21
1,01002,84,01,L'Abergement-de-Varey,14,19,20,14,11,7,...,12,12,91,8,5,16,18,13,17,14
2,01004,84,01,Ambérieu-en-Bugey,833.778507539282,992.962423609689,773.079155075357,611.095184700315,1470.39101358599,1079.70432852828,...,374.522477434457,507.55702096894,5975.84010149909,1366.46149390245,403.856706954946,1373.36573978546,1098.85402811873,718.709133828151,565.031961733504,449.56103717585


In [29]:
df_dipl = df_dipl_raw.copy()

# Renommer colonne code commune
df_dipl = df_dipl.rename(columns={'CODGEO': 'code_commune'})
df_dipl['code_commune'] = df_dipl['code_commune'].astype(str).str.zfill(5)
df_dipl = df_dipl[df_dipl['code_commune'].isin(codes_pc)].copy()
df_dipl['annee'] = 2022

# Mappage colonnes diplômes
dipl_mapping = {
    'P22_NSCOL15P':          'nscol15p',
    'P22_NSCOL15P_DIPLMIN':  'sans_diplome',
    'P22_NSCOL15P_BEPC':     'bepc_brevet',
    'P22_NSCOL15P_CAPBEP':   'cap_bep',
    'P22_NSCOL15P_BAC':      'bac',
    'P22_NSCOL15P_SUP2':     'sup_bac2',
    'P22_NSCOL15P_SUP34':    'sup_bac34',
    'P22_NSCOL15P_SUP5':     'sup_bac5',
}

df_dipl_silver = df_dipl[['code_commune', 'annee']].copy()
for bronze_col, silver_col in dipl_mapping.items():
    if bronze_col in df_dipl.columns:
        df_dipl_silver[silver_col] = pd.to_numeric(df_dipl[bronze_col], errors='coerce')
    else:
        print(f"WARN: colonne {bronze_col} absente")
        df_dipl_silver[silver_col] = np.nan

df_dipl_silver['annee'] = df_dipl_silver['annee'].astype('Int16')
df_dipl_silver = df_dipl_silver.drop_duplicates(subset=['code_commune', 'annee'])

print(f"Diplômes Silver : {len(df_dipl_silver)} communes")
df_dipl_silver.head(3)

Diplômes Silver : 123 communes


,code_commune,annee,nscol15p,sans_diplome,bepc_brevet,cap_bep,bac,sup_bac2,sup_bac34,sup_bac5
29194,75056,2022,1.543807e+06,177701.904719,51772.893309,115690.648916,188395.049144,121234.527881,235649.793979,653361.743737
34441,92002,2022,4.497512e+04,4552.197651,1638.384448,4649.106552,5522.232535,4736.512769,6510.999038,17365.686640
34442,92004,2022,6.337710e+04,8836.839664,1870.497327,6819.999094,8658.087119,6249.284451,8612.345547,22330.051333


In [30]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE silver.diplomes"))

df_dipl_silver.to_sql('diplomes', engine, schema='silver', if_exists='append', index=False, method='multi', chunksize=1000)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM silver.diplomes")).scalar()
print(f"silver.diplomes : {n} lignes")

silver.diplomes : 123 lignes


## 8. Validation finale

In [31]:
validation_queries = {
    'referentiel_communes': 'SELECT COUNT(*), COUNT(DISTINCT code_dep) FROM silver.referentiel_communes',
    'elections':            'SELECT COUNT(*), MIN(annee), MAX(annee), COUNT(DISTINCT code_commune) FROM silver.elections',
    'naissances':           'SELECT COUNT(*), MIN(annee), MAX(annee), COUNT(DISTINCT code_commune) FROM silver.naissances',
    'deces':                'SELECT COUNT(*), MIN(annee), MAX(annee), COUNT(DISTINCT code_commune) FROM silver.deces',
    'revenus':              'SELECT COUNT(*), MIN(annee), MAX(annee), COUNT(DISTINCT code_commune) FROM silver.revenus',
    'csp':                  'SELECT COUNT(*), COUNT(DISTINCT code_commune) FROM silver.csp',
    'diplomes':             'SELECT COUNT(*), COUNT(DISTINCT code_commune) FROM silver.diplomes',
}

print("=" * 60)
print("VALIDATION SILVER LAYER")
print("=" * 60)

with engine.connect() as conn:
    for table, query in validation_queries.items():
        result = conn.execute(text(query)).fetchone()
        print(f"\n{table}:")
        print(f"  -> {result}")

print("\n" + "=" * 60)
print("ETL Bronze -> Silver terminé avec succès !")
print("=" * 60)

VALIDATION SILVER LAYER

referentiel_communes:
  -> (144, 4)

elections:
  -> (1517115, 1999, 2024, 124)

naissances:
  -> (2431, 2008, 2024, 143)

deces:
  -> (2431, 2008, 2024, 143)

revenus:
  -> (0, None, None, 0)

csp:
  -> (143, 143)

diplomes:
  -> (123, 123)

ETL Bronze -> Silver terminé avec succès !
